In [ ]:
from pathlib import Path
from docking_pipeline import AnalysisConfig, build_scored_pose_residue_table_pipeline, run_free_energy_pipeline

In [ ]:
###   Raw extraction from pdbs

config_raw = AnalysisConfig(
    base_dir=Path("."),
    output_dir=Path("results"),
    poses_dir=Path("all_poses"),
    selected_pose_ids=best_select,  # your original list
    prodigy_executable="prodigy_lig",
)

scored_df = build_scored_pose_residue_table_pipeline(config_raw)
scored_df.head()

In [ ]:
###   Free-energy analysis

config_analysis = AnalysisConfig(
    base_dir=Path("."),
    output_dir=Path("results"),
    pose_residue_contact_file=Path("results/pose_residue_contacts.csv"),
    pose_score_file=Path("results/prodigy_scores.csv"),
    selected_pose_ids=best_select,
    free_energy_method="notebook_basic",
)

results = run_free_energy_pipeline(config_analysis)

enthalpy_df = results["enthalpy"]
entropy_df = results["entropy"]
free_energy_df = results["free_energy"]

In [ ]:
###   Inspection

free_energy_df.sort_values("free_energy").head(20)

In [ ]:
import matplotlib.pyplot as plt

plt.plot(free_energy_df["residue_id"], free_energy_df["free_energy"])
plt.xlabel("Residue")
plt.ylabel("Free Energy")
plt.title("Residue Free Energy Profile")
plt.show()

In [ ]:
###   Debugging/sanity checks

# Check repartition consistency
df = results["pose_residue_contacts"]

check = df["pose_energy"] / df["n_contacts_in_pose"]
print((df["reparted_energy"] - check).abs().max())

In [ ]:
# Check entropy normalization
entropy_df["pose_fraction"].sum()